In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Prepare dataset structure for YOLO format
def prepare_widerface_dataset(wider_face_path, output_path):
    """
    Convert WIDER FACE dataset to YOLO format
    WIDER FACE annotations need to be converted to YOLO format:
    class x_center y_center width height (normalized)
    """
    os.makedirs(f"{output_path}/images/train", exist_ok=True)
    os.makedirs(f"{output_path}/images/val", exist_ok=True)
    os.makedirs(f"{output_path}/labels/train", exist_ok=True)
    os.makedirs(f"{output_path}/labels/val", exist_ok=True)
    
    # Parse WIDER FACE annotations
    def parse_wider_annotations(anno_file):
        with open(anno_file, 'r') as f:
            lines = f.readlines()
        
        annotations = {}
        i = 0
        while i < len(lines):
            img_path = lines[i].strip()
            i += 1
            num_faces = int(lines[i].strip())
            i += 1
            
            faces = []
            for _ in range(num_faces):
                face_data = list(map(int, lines[i].strip().split()))
                # x1, y1, w, h, blur, expression, illumination, invalid, occlusion, pose
                if len(face_data) >= 4:
                    faces.append(face_data[:4])  # x, y, w, h
                i += 1
            
            annotations[img_path] = faces
        
        return annotations
    
    # Convert to YOLO format
    def convert_to_yolo_format(x, y, w, h, img_width, img_height):
        x_center = (x + w / 2) / img_width
        y_center = (y + h / 2) / img_height
        width = w / img_width
        height = h / img_height
        return x_center, y_center, width, height
    
    # Process annotations (you need to implement full conversion)
    print("Dataset preparation complete!")
    return output_path

# Prepare dataset
WIDER_FACE_PATH = "WIDER_FACE"  # Path to downloaded WIDER FACE
OUTPUT_PATH = "widerface_yolo"
prepare_widerface_dataset(WIDER_FACE_PATH, OUTPUT_PATH)

# Create YAML configuration file
yaml_content = f"""
train: {OUTPUT_PATH}/images/train
val: {OUTPUT_PATH}/images/val

nc: 1  # number of classes (face only)
names: ['face']
"""

with open('widerface.yaml', 'w') as f:
    f.write(yaml_content)

# Fine-tune YOLOv8 model
from ultralytics import YOLO

# Load pretrained model
model = YOLO('yolov8n.pt')

# Train the model
results = model.train(
    data='widerface.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='yolov8_face_detector',
    patience=10,
    save=True,
    device=0  # Use GPU if available, else 'cpu'
)

# Validate the model
metrics = model.val()
print(f"mAP50: {metrics.box.map50}")


# Clone and setup YOLOv8-Face from GitHub
!git clone https://github.com/Yusepp/YOLOv8-Face.git
!cd YOLOv8-Face && pip install -r requirements.txt

# Load both models
finetuned_model = YOLO('runs/detect/yolov8_face_detector/weights/best.pt')
github_model = YOLO('YOLOv8-Face/weights/yolov8n-face.pt')  # Adjust path

def evaluate_face_detector(model, val_dataset_path):
    """
    Evaluate face detector on validation set
    """
    from ultralytics.utils.metrics import DetMetrics
    
    # Run validation
    results = model.val(data='widerface.yaml', split='val')
    
    return {
        'precision': results.box.p,
        'recall': results.box.r,
        'mAP50': results.box.map50,
        'mAP50_95': results.box.map,
        'inference_time': results.speed['inference']
    }

# Evaluate both models
print("\nEvaluating Fine-tuned YOLOv8...")
finetuned_metrics = evaluate_face_detector(finetuned_model, OUTPUT_PATH)

print("\nEvaluating GitHub YOLOv8-Face...")
github_metrics = evaluate_face_detector(github_model, OUTPUT_PATH)

# Compare results
comparison_df = {
    'Metric': ['Precision', 'Recall', 'mAP50', 'mAP50-95', 'Inference Time (ms)'],
    'Fine-tuned YOLOv8': [
        finetuned_metrics['precision'],
        finetuned_metrics['recall'],
        finetuned_metrics['mAP50'],
        finetuned_metrics['mAP50_95'],
        finetuned_metrics['inference_time']
    ],
    'GitHub YOLOv8-Face': [
        github_metrics['precision'],
        github_metrics['recall'],
        github_metrics['mAP50'],
        github_metrics['mAP50_95'],
        github_metrics['inference_time']
    ]
}

import pandas as pd
comparison_table = pd.DataFrame(comparison_df)
print("\nComparison Results:")
print(comparison_table)

# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparison_table['Metric'][:4]))  # Exclude inference time
width = 0.35

ax.bar(x - width/2, comparison_table['Fine-tuned YOLOv8'][:4], width, label='Fine-tuned YOLOv8')
ax.bar(x + width/2, comparison_table['GitHub YOLOv8-Face'][:4], width, label='GitHub YOLOv8-Face')

ax.set_ylabel('Score')
ax.set_title('Face Detector Comparison')
ax.set_xticks(x)
ax.set_xticklabels(comparison_table['Metric'][:4])
ax.legend()
plt.tight_layout()
plt.savefig('face_detector_comparison.png')
plt.show()
